F1 Circuit Complexity Index
--------
Formula 1 is the peak of motorsport where only the best drivers from around the world are allowed behind the wheel and onto the track. Each track is unique and has its own set of technical and physical challenges the drivers must adapt to. This analysis looks to develop a composite indicator designed to quantify the difficulty of all circuits.

## Finding the data

To create this index I am going to need to combine a few datasets. I will need data about the circuits that includes their length, number of corners, height elevation, top speed etc. To better show how difficult a track is I will also seek out a dataset containing all safety cars deployed and their reason for each track. 

I am interested to see how the safety car dataset will influence the result as you could argue that a track that has more safety cars is more difficult. Finally I will attempt to find a dataset containing time penalties given to drivers, specifically track limit penalties. This occurs when all four tyres cross the white line markings of the track edge, possibly indicating a higher difficulty.


- Data Sources
1. <a href="https://mintlify.wiki/TracingInsights/RaceData/introduction">RaceData</a>
2. <a href="https://www.kaggle.com/datasets/kishan305/formula-1-circuits-1950-present">Kaggle - Formula 1 Circuits (1950 - Present)</a>


**Race Data** is a comprehensive Formula 1 race data from 1950 to present. They provide 18 data tables that will provide me with safety car, red flags and fatal accident data.

**Kaggle** Although RaceData does provide a dataset on all circuits I have opted to use this kaggle dataset as there is more technical information about each track in it.


TODO: Talk about low amount of data rows but lots of data points, I have captured nearly all of the circuits and it is about data quality not quantity

In [341]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

circuit_metrics = pd.read_csv("data/all_f1_circuits.csv")
circuit_metrics.head()

,Circuit,City,Country,Track Length (km),Turns,Direction,Circuit Type,First Grand Prix,Last Grand Prix,Races,Best Lap Timing,Best Lap Driver,Best Lap Year,Best Lap Time
0,Brands Hatch,Fawkham,United Kingdom,4.207,9,Clockwise,Race,1964 British Grand Prix,1986 British Grand Prix,14,1:09.593,Nigel Mansell,1986,69.593
1,AVUS,Berlin,Germany,8.300,4,Anti clockwise,Road,1959 German Grand Prix,1959 German Grand Prix,1,2:04.500,Tony Brooks,1959,124.500
2,Autodromo Internazionale Enzo e Dino Ferrari,Imola,Italy,4.909,19,Anti clockwise,Race,1980 Italian Grand Prix,2025 Emilia Romagna Grand Prix,32,1:15.484,Lewis Hamilton,2020,75.484
3,Autódromo Juan y Oscar Gálvez,Buenos Aires,Argentina,4.259,19,Clockwise,Race,1953 Argentine Grand Prix,1998 Argentine Grand Prix,20,1:11.220,Emerson Fittipaldi,1973,71.220
4,Autodromo Internazionale del Mugello,Scarperia e San Piero,Italy,5.245,15,Clockwise,Race,2020 Tuscan Grand Prix,2020 Tuscan Grand Prix,1,1:18.833,Lewis Hamilton,2020,78.833


In [342]:
safety_cars = pd.read_csv("data/safety_cars.csv")
safety_cars.head()

,Race,Cause,Deployed,Retreated,FullLaps
0,1973 Canadian Grand Prix,Accident,33,39.0,5
1,1993 Brazilian Grand Prix,Accident/Rain,29,38.0,8
2,1993 British Grand Prix,Stranded car,38,40.0,1
3,1994 San Marino Grand Prix,Accident,1,6.0,4
4,1995 Belgian Grand Prix,Rain,28,33.0,4


In [343]:
red_flags = pd.read_csv("data/red_flags.csv")
red_flags.head()

,Race,Lap,Resumed,Incident,Excluded
0,1950 Indianapolis 500,138,N,Rain.,NaN
1,1971 Canadian Grand Prix,64,N,Mist.,NaN
2,1973 British Grand Prix,2,Y,"Crash involving Jody Scheckter, Jean-Pierre Be...","Jody Scheckter, Jean-Pierre Beltoise, George F..."
3,1974 Brazilian Grand Prix,32,N,Rain.,NaN
4,1975 Spanish Grand Prix,29,N,Crash of Rolf Stommelen which killed five spec...,NaN


In [344]:
fatal_accidents_drivers = pd.read_csv("data/fatal_accidents_drivers.csv")
fatal_accidents_drivers.head()

,Driver,Age,Date Of Accident,Event,Car,Session
0,Cameron Earl,29.0,6/18/52,NaN,ERA,Test
1,Chet Miller,50.0,5/15/53,1953 Indianapolis 500,Kurtis Kraft,Practice
2,Charles de Tornaco,26.0,9/18/53,1953 Modena Grand Prix,Ferrari,Practice
3,Onofre Marimón,30.0,7/31/54,1954 German Grand Prix,Maserati,Practice
4,Mario Alborghetti,26.0,4/11/55,1955 Pau Grand Prix,Maserati,Race


In [345]:
## Added during selecting variables
circuit_info = pd.read_csv("data/circuits.csv")
circuit_info.head()

,circuitId,circuitRef,name,location,country,lat,lng,alt,url
0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10,http://en.wikipedia.org/wiki/Melbourne_Grand_P...
1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,18,http://en.wikipedia.org/wiki/Sepang_Internatio...
2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,7,http://en.wikipedia.org/wiki/Bahrain_Internati...
3,4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57000,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcel...
4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,130,http://en.wikipedia.org/wiki/Istanbul_Park


Selecting Variables
------

To select my variables I am using excel to view the raw data. What is referenced will be from those observations but I will show what I am talking about through visualization here. As the data is coming from 2 different sources there are likely some discrepancies hidden in them. For example the above head of fatal_accidents_driver and red_flags have NaN values, some tracks are describe using a slightly different name. 


Something important I have to consider to ensure this index is accurate is what is the cut off year before I start excluding race data from my data set. I think this because if you look back at what the level of safety standards were back in 1950 to 1980 they were basically non-existent. Only until the mid 90s did the FIA enforce strict rules on the teams and cars. With this in mind and before fully getting into this data i think it would be best to use race data from the year 2000 until present.

In [346]:
master_data = []

#### All Circuits

Although the race name is not important for the index result, it will be a key part of linking the data together. Something I immediately noticed is that all_f1_circuits.csv  uses the proper circuit name and the others use a format of - $Year + Country + Grand Prix$. In all_circuits_f1 I am given the city and the country but using this as a link will fail on races like British and Italian as these are now the country names. To get passed this im going to use the circuits data provided by RaceData which contains the name of the circuit. 

By merging these two together I can also include the circuitId provided by circuits.csv to then use as a link to all the other datasets from RaceData making it much easier to add data to my problem.

In [347]:
# sorted by name to show how they line up
sorted_metrics = circuit_metrics.sort_values(by='Circuit')
sorted_info = circuit_info.sort_values(by='name')

print("(Kaggle) all_f1_circuits.csv description of track")
print(list(sorted_metrics.Circuit.values))
print(f"Length - {len(sorted_metrics)}")
print("\n(RaceData) circuits.csv description of track")
print(list(sorted_info.name.values))
print(f"Length - {len(sorted_info)}")

(Kaggle) all_f1_circuits.csv description of track
['AVUS', 'Adelaide Street Circuit', 'Ain-Diab Circuit', 'Aintree Motor Racing Circuit', 'Algarve International Circuit', 'Anderstorp Raceway', 'Autodromo Internazionale Enzo e Dino Ferrari', 'Autodromo Internazionale del Mugello', 'Autodromo Nazionale Monza', 'Autódromo Hermanos Rodríguez', 'Autódromo Internacional Nelson Piquet', 'Autódromo José Carlos Pace', 'Autódromo Juan y Oscar Gálvez', 'Autódromo do Estoril', 'Bahrain International Circuit', 'Baku City Circuit', 'Brands Hatch', 'Buddh International Circuit', 'Bugatti Circuit', 'Caesars Palace', 'Canadian Tire Motorsport Park', 'Charade Circuit', 'Circuit Bremgarten', 'Circuit Gilles Villeneuve', 'Circuit Mont-Tremblant', 'Circuit Park Zandvoort', 'Circuit Paul Ricard', 'Circuit Zolder', 'Circuit de Barcelona-Catalunya', 'Circuit de Monaco', 'Circuit de Nevers Magny-Cours', 'Circuit de Spa-Francorchamps', 'Circuit of the Americas', 'Circuito da Boavista', 'Circuito de Jerez', 'Cir

There is 1 race missing from the all_f1_circuits and also some of the circuits still come under a different name. I will find the ones that match and review the misaligned ones using python sets. 

In [348]:
# first I want to check how many of the circuit names match between the datasets
matching_circuits = []
match_count = 0
bad_count = 0
print(len(circuit_metrics))
print(len(circuit_info))

# create sets of the unique names in both columns
circuit_names1 = set(circuit_metrics['Circuit'])
circuit_names2 = set(circuit_info['name']) # Replace with your actual column name

# using intersection to get a list of matching items between the sets - https://www.w3schools.com/python/ref_set_intersection.asp
matches = circuit_names1.intersection(circuit_names2)
metrics_mismatches = circuit_names1.difference(circuit_names2)
info_mismatches = circuit_names2.difference(circuit_names1)

print(metrics_mismatches)
print(info_mismatches)
print(f"Matches: {len(matches)}")
print(f"Circuit Metrics: {len(metrics_mismatches)}")
print(f"Circuit Info: {len(info_mismatches)}")

77
78
{'Watkins Glen International', 'Melbourne Grand Prix Circuit', 'Circuito de Monsanto', 'Anderstorp Raceway', 'Algarve International Circuit', 'Canadian Tire Motorsport Park', 'Circuito de Montjuïc', 'Aintree Motor Racing Circuit', 'Lusail International Circuit', 'Korea International Circuit', 'Phoenix Street Circuit', 'Autodromo Nazionale Monza', 'Autodromo Internazionale Enzo e Dino Ferrari', 'Kyalami Racing Circuit', 'Pedralbes Circuit', 'Bugatti Circuit', 'Ain-Diab Circuit', 'Caesars Palace', 'Circuit Zolder', 'Circuito del Jarama'}
{'Zolder', 'Monsanto Park Circuit', 'Jarama', 'Watkins Glen', 'Madring', 'Circuit de Pedralbes', 'Albert Park Grand Prix Circuit', 'Las Vegas Strip Street Circuit', 'Aintree', 'Losail International Circuit', 'Kyalami', 'Montjuïc', 'Autodromo Enzo e Dino Ferrari', 'Mosport International Raceway', 'Ain Diab', 'Autodromo Nazionale di Monza', 'Le Mans', 'Phoenix street circuit', 'Scandinavian Raceway', 'Korean International Circuit', 'Autódromo Interna

In [349]:
print(sorted(metrics_mismatches))
print(sorted(info_mismatches))

['Ain-Diab Circuit', 'Aintree Motor Racing Circuit', 'Algarve International Circuit', 'Anderstorp Raceway', 'Autodromo Internazionale Enzo e Dino Ferrari', 'Autodromo Nazionale Monza', 'Bugatti Circuit', 'Caesars Palace', 'Canadian Tire Motorsport Park', 'Circuit Zolder', 'Circuito de Monsanto', 'Circuito de Montjuïc', 'Circuito del Jarama', 'Korea International Circuit', 'Kyalami Racing Circuit', 'Lusail International Circuit', 'Melbourne Grand Prix Circuit', 'Pedralbes Circuit', 'Phoenix Street Circuit', 'Watkins Glen International']
['Ain Diab', 'Aintree', 'Albert Park Grand Prix Circuit', 'Autodromo Enzo e Dino Ferrari', 'Autodromo Nazionale di Monza', 'Autódromo Internacional do Algarve', 'Circuit de Pedralbes', 'Jarama', 'Korean International Circuit', 'Kyalami', 'Las Vegas Strip Street Circuit', 'Le Mans', 'Losail International Circuit', 'Madring', 'Monsanto Park Circuit', 'Montjuïc', 'Mosport International Raceway', 'Phoenix street circuit', 'Scandinavian Raceway', 'Watkins Gle

Looking at the circuits the easiest thing I can do here is create a mapping dict that I can refer back to in a loop to get the correct circuit. Looking through the lists I have found that Madring is the missing circuit. I have opted to exclude it from the data.

In [350]:
# all_f1_circuit.csv as key | circuits.csv as value
circuit_mapping = {
    'Ain-Diab Circuit': 'Ain Diab',
    'Aintree Motor Racing Circuit': 'Aintree',
    'Algarve International Circuit': 'Autódromo Internacional do Algarve',
    'Anderstorp Raceway': 'Scandinavian Raceway',
    'Autodromo Internazionale Enzo e Dino Ferrari': 'Autodromo Enzo e Dino Ferrari',
    'Autodromo Nazionale Monza': 'Autodromo Nazionale di Monza',
    'Bugatti Circuit': 'Le Mans',
    'Caesars Palace': 'Las Vegas Strip Street Circuit', 
    'Canadian Tire Motorsport Park': 'Mosport International Raceway',
    'Circuit Zolder': 'Zolder',
    'Circuito de Monsanto': 'Monsanto Park Circuit',
    'Circuito de Montjuïc': 'Montjuïc',
    'Circuito del Jarama': 'Jarama',
    'Korea International Circuit': 'Korean International Circuit',
    'Kyalami Racing Circuit': 'Kyalami',
    'Lusail International Circuit': 'Losail International Circuit',
    'Melbourne Grand Prix Circuit': 'Albert Park Grand Prix Circuit',
    'Pedralbes Circuit': 'Circuit de Pedralbes',
    'Phoenix Street Circuit': 'Phoenix street circuit',
    'Watkins Glen International': 'Watkins Glen'
}

In [ ]:
uncombined_metrics = circuit_metrics[['Circuit', 'Track Length (km)', 'Turns', 'Direction', 'Circuit Type']]
uncombined_info = circuit_info[['circuitId', 'name', 'country']]
combined_data = []

for index_m, row_m in uncombined_metrics.iterrows():
    circuit_name = row_m['Circuit']
    
    target_search_name = circuit_mapping.get(circuit_name, circuit_name)
    
    found = False
    for index_i, row_i in uncombined_info.iterrows():
        if row_i['name'] == target_search_name:
            combined_row = {
                'Circuit': circuit_name,
                'Track Length (km)': row_m['Track Length (km)'],
                'Turns': row_m['Turns'],
                'Direction': row_m['Direction'],
                'Circuit Type': row_m['Circuit Type'],
                'circuitId': row_i['circuitId'],
                'country': row_i['country']
            }
            master_data.append(combined_row)
            found = True
            # print(f"match: {circuit_name} - {target_search_name}")
            break
            
    # if not found:
        # print(f"no match: {circuit_name} - looking for '{target_search_name}')")

combined_circuits = pd.DataFrame(master_data)
combined_circuits.head()
print(len(combined_circuits))

77


#### Now that I have combined these two datasets the rest can be combined using circuitId taken from the RaceData dataset

#### Safety Cars

In [352]:
safety_cars.head()

,Race,Cause,Deployed,Retreated,FullLaps
0,1973 Canadian Grand Prix,Accident,33,39.0,5
1,1993 Brazilian Grand Prix,Accident/Rain,29,38.0,8
2,1993 British Grand Prix,Stranded car,38,40.0,1
3,1994 San Marino Grand Prix,Accident,1,6.0,4
4,1995 Belgian Grand Prix,Rain,28,33.0,4


To my surprise, even though the safety_cars.csv comes from the same source that uses circuitId, the only useful variable to match is the race name. This race name does not match what I have just done above...

There are 3 variables that I am interested in from this set.

- Cause
- Deployed
- FullLaps

#### Red Flags

#### Fatal Accidents (Drivers)